# LA 02b — First-touch versus second-touch aggressive flow and zone volume

**Research-only causal feature enrichment; no trade P&L.** This module *does not select new signals*. It uses the 452 price-only A02 candidates and reconstructs trade flow during each actual touch candle, plus historical price-volume information available before the first touch. Trading rules, expected target/stop probabilities, fee assumptions and thresholds remain **unfrozen**.

The local Binance USD-M BTC/ETH aggregated trade files are scanned in bounded batches. Data are not written back to the raw archive. Input events are loaded from `02/02_price_only_candidates.csv`, and results are written to `02b/`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

REPO = Path.cwd().resolve()
while not (REPO / "src" / "data_engineering").exists():
    if REPO == REPO.parent:
        raise RuntimeError("Launch Jupyter in repository or a subdirectory")
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.data_engineering.modeling.liquidity_absorption_stage_a import binance_root
from src.data_engineering.modeling.liquidity_absorption_stage_a02b import (
    Config, LA_A02B_VERSION, run_a02b,
)
assert LA_A02B_VERSION == "LA-A02b-0.1.0", "Pull repository and restart kernel"
DATA_ROOT = binance_root(REPO)
OUT = REPO / "notebooks" / "Liquidity_Absorption" / "02b"
CFG = Config()
print("Version:", LA_A02B_VERSION, "| Root:", DATA_ROOT, "| Output:", OUT)
print("Mode: EXPLORATORY FEATURE-ONLY; no forward outcomes / P&L")

Version: LA-A02b-0.1.0 | Root: D:\MarketData\Binance | Output: D:\Projects\deribit_market_data\notebooks\Liquidity_Absorption\02b
Mode: EXPLORATORY FEATURE-ONLY; no forward outcomes / P&L


In [2]:
decision = run_a02b(DATA_ROOT, REPO, OUT, cfg=CFG)
decision

{'version': 'LA-A02b-0.1.0',
 'status': 'FEATURE_EXTRACTION_ONLY',
 'candidate_count': 452,
 'source_a02_candidate_count': 452,
 'coverage': [{'symbol': 'BTCUSDT',
   'events': 212,
   'touch_flow_present': 212,
   'missing_touch_day_events': 0,
   'pre_first_profile_present': 212,
   'pre_first_profile_missing_day': 0,
   'zero_first_zone_quote': 0,
   'zero_second_zone_quote': 0,
   'undefined_touch_volume_ratio': 0,
   'undefined_late_early_dwell_ratio': 0,
   'median_second_touch_reconciliation_error': 5.712255503646706e-16,
   'p95_second_touch_reconciliation_error': 7.811667347950577e-05},
  {'symbol': 'ETHUSDT',
   'events': 240,
   'touch_flow_present': 240,
   'missing_touch_day_events': 0,
   'pre_first_profile_present': 240,
   'pre_first_profile_missing_day': 0,
   'zero_first_zone_quote': 0,
   'zero_second_zone_quote': 0,
   'undefined_touch_volume_ratio': 0,
   'undefined_late_early_dwell_ratio': 0,
   'median_second_touch_reconciliation_error': 4.292583931286804e-16,
  

In [3]:
features = pd.read_csv(OUT / "02b_touch_features.csv")
coverage = pd.read_csv(OUT / "02b_coverage.csv")
display(coverage)
print("Events:", len(features), "| Presence:", features.touch_status.value_counts(dropna=False).to_dict())
assert len(features) == decision["source_a02_candidate_count"], "A02 source rows changed"
assert features.a02_candidate_row.is_unique, "Duplicated candidate identity"
for label, cols in [
    ("touch observations", ["symbol", "side", "first_touch_close", "second_touch_close", "first_zone_quote", "second_zone_quote", "zone_quote_touch_ratio", "first_opposing_zone_share", "second_opposing_zone_share"]),
    ("relative to time around level", ["symbol", "side", "intersecting_1m_candles", "early_zone_quote_per_intersecting_1m", "late_zone_quote_per_intersecting_1m", "late_early_zone_rate_ratio", "zone_active_print_seconds_between_touches"]),
    ("pre-first-touch profile", ["symbol", "level_price", "pre_first_profile_status", "pre_first_poc_price", "pre_first_poc_distance_bp", "pre_first_profile_zone_share"]),
]:
    print("\n",label)
    display(features[cols].head(10))
print("\nMissing ratio counts:")
print(features[["zone_quote_touch_ratio", "late_early_zone_rate_ratio", "pre_first_poc_price"]].isna().sum())

,symbol,events,touch_flow_present,missing_touch_day_events,pre_first_profile_present,pre_first_profile_missing_day,zero_first_zone_quote,zero_second_zone_quote,undefined_touch_volume_ratio,undefined_late_early_dwell_ratio,median_second_touch_reconciliation_error,p95_second_touch_reconciliation_error
0,BTCUSDT,212,212,0,212,0,0,0,0,0,5.712256e-16,0.000078
1,ETHUSDT,240,240,0,240,0,0,0,0,0,4.292584e-16,0.000497


Events: 452 | Presence: {'PRESENT': 452}

 touch observations


,symbol,side,first_touch_close,second_touch_close,first_zone_quote,second_zone_quote,zone_quote_touch_ratio,first_opposing_zone_share,second_opposing_zone_share
0,BTCUSDT,short,2026-07-01 02:29:00+00:00,2026-07-01 02:33:00+00:00,8.605479e+05,3.622141e+05,0.420911,0.274914,0.553818
1,BTCUSDT,long,2026-07-01 12:41:00+00:00,2026-07-01 12:44:00+00:00,6.664700e+06,3.358163e+06,0.503873,0.529467,0.556281
2,BTCUSDT,long,2026-07-01 15:59:00+00:00,2026-07-01 16:11:00+00:00,1.132776e+06,4.034734e+06,3.561811,0.279801,0.706300
3,BTCUSDT,short,2026-07-02 10:43:00+00:00,2026-07-02 10:55:00+00:00,5.157327e+06,2.954606e+05,0.057289,0.491171,0.457564
4,BTCUSDT,short,2026-07-02 11:10:00+00:00,2026-07-02 11:24:00+00:00,1.494692e+06,2.036021e+06,1.362168,0.350558,0.436420
5,BTCUSDT,short,2026-07-02 11:25:00+00:00,2026-07-02 11:39:00+00:00,7.046682e+05,3.349491e+06,4.753288,0.231008,0.577860
6,BTCUSDT,short,2026-07-02 12:11:00+00:00,2026-07-02 12:18:00+00:00,2.011516e+06,2.440140e+06,1.213085,0.238029,0.671870
7,BTCUSDT,long,2026-07-03 04:10:00+00:00,2026-07-03 04:19:00+00:00,1.330941e+06,2.140604e+05,0.160834,0.671359,0.215765
8,BTCUSDT,long,2026-07-04 10:52:00+00:00,2026-07-04 11:03:00+00:00,1.360126e+05,7.360254e+05,5.411451,0.329660,0.338072
9,BTCUSDT,short,2026-07-04 13:23:00+00:00,2026-07-04 13:26:00+00:00,1.763170e+06,1.397043e+06,0.792347,0.698762,0.332497



 relative to time around level


,symbol,side,intersecting_1m_candles,early_zone_quote_per_intersecting_1m,late_zone_quote_per_intersecting_1m,late_early_zone_rate_ratio,zone_active_print_seconds_between_touches
0,BTCUSDT,short,3,8.605479e+05,1.983925e+07,23.054210,31
1,BTCUSDT,long,3,6.664700e+06,3.705735e+06,0.556024,90
2,BTCUSDT,long,9,2.804009e+06,3.384216e+06,1.206920,298
3,BTCUSDT,short,2,5.157327e+06,2.954606e+05,0.057289,12
4,BTCUSDT,short,5,6.804002e+06,2.036021e+06,0.299239,88
5,BTCUSDT,short,5,5.673290e+05,2.386205e+06,4.206033,54
6,BTCUSDT,short,5,2.011516e+06,5.489396e+06,2.728985,87
7,BTCUSDT,long,5,3.849495e+06,8.687434e+05,0.225677,117
8,BTCUSDT,long,2,1.360126e+05,7.360254e+05,5.411451,14
9,BTCUSDT,short,2,1.763170e+06,1.397043e+06,0.792347,61



 pre-first-touch profile


,symbol,level_price,pre_first_profile_status,pre_first_poc_price,pre_first_poc_distance_bp,pre_first_profile_zone_share
0,BTCUSDT,58991.6,PRESENT,59000.473254,1.504156,0.147394
1,BTCUSDT,58350.9,PRESENT,58401.757006,8.715719,0.016351
2,BTCUSDT,60099.1,PRESENT,60102.085425,0.496750,0.112003
3,BTCUSDT,61322.0,PRESENT,61248.759445,-11.943602,0.000000
4,BTCUSDT,61330.0,PRESENT,61248.759445,-13.246463,0.011574
5,BTCUSDT,61322.0,PRESENT,61248.759445,-11.943602,0.095842
6,BTCUSDT,61322.0,PRESENT,61267.135910,-8.946885,0.045759
7,BTCUSDT,61327.2,PRESENT,61371.373406,7.202906,0.053446
8,BTCUSDT,62436.2,PRESENT,62461.015825,3.974589,0.098701
9,BTCUSDT,62671.5,PRESENT,62736.436003,10.361329,0.084051



Missing ratio counts:
zone_quote_touch_ratio        0
late_early_zone_rate_ratio    0
pre_first_poc_price           0
dtype: int64


## Interpretation before LA 03

1. **First/second touches** are exactly the completed 1-minute rejection candles already selected in A02, and the order flow uses only trades strictly before the signal timestamp. No retroactive stop changes or entries have been modeled.
2. **Near-level volume** uses the *actual aggregated trade price* within `[level − halfwidth, level + halfwidth]`, not 1bp midpoint bins. This is only executed volume, not passive order-book depth or confirmed iceberg activity.
3. **Dwell** is measured as the count of 1-minute candles whose OHLC intersects the zone. It is a price-visitation proxy, not seconds actually spent at that price. Separate “active print seconds” count seconds with a trade **at the level**, not continuous market occupation.
4. **First versus second volume** is normalized both per touch candle (fixed duration) and as a fraction of each full minute's executed quote volume. Time-between-touch early/late intensity is divided by the count of minute bars intersecting the zone, not by the entire elapsed gap.
5. **Zero first-touch volume** means the growth ratio remains undefined (`NaN`), rather than being replaced by zero or infinity. Read `touch_status` and `pre_first_profile_status` separately. Files missing before first touch do not erase available two-touch features.
6. The **pre-first point of control (POC)** is built from an untuned trailing 30-minute profile ending *before the first rejection candle starts*. It is only a reference attribute; A02b does **not** create events at POC or low-volume nodes.
7. Look at both first and second candle `*_trade_kline_quote_error` columns; large errors suggest a trade-archive completeness or alignment issue.

**Next:** commit the executed notebook plus all files in `notebooks/Liquidity_Absorption/02b/`, then request a review. A03 requires a separate, owner-reviewed specification for executable entry, stop/BE/trailing timing, causal profit-target discovery, and Binance-specific trading costs.